# Chapter 27 — Transformer III: Vision Transformers, VQA, Large Language Models & RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part4_transformers_genai/ch27_transformer_iii/ch27_notebook.ipynb)

Companion notebook for **Chapter 27** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Cut an image into fixed-size patches and it becomes a sequence, which is all a Transformer needs (the Vision Transformer).
- Visual question answering combines an image encoder, a question encoder, a fusion step and an answer generator.
- Large language models are simply pre-trained language models scaled to billions of parameters.
- A foundation model is pre-trained once and fine-tuned many times for different downstream tasks.
- Retrieval Augmented Generation retrieves supporting documents and places them in the prompt, which grounds answers and reduces hallucination.

## Example 1: Vision Transformer Patch Embedding

This performs the four steps that turn an image into a sequence: split into patches, flatten and linearly project them, prepend the learnable [CLS] token whose final output is used for classification, and add position embeddings. The table at the end shows the trade-off in patch size - smaller patches give more tokens and finer detail at higher computational cost.

In [1]:
# Chapter 27 - Example 1: Vision Transformer (ViT) patch embedding
import numpy as np
np.random.seed(0)

print("=== Vision Transformer (ViT): 'An image is worth 16x16 words' ===\n")
print("Idea: split the image into fixed-size patches, flatten each patch,")
print("linearly project it -> now the image IS a sequence, so a Transformer")
print("encoder can process it exactly like a sentence.\n")

H = W = 32          # image size
C = 3               # channels (RGB)
P = 8               # patch size
d = 16              # embedding dimension

image = np.random.rand(H, W, C)
print(f"Input image : {H}x{W}x{C}")
print(f"Patch size  : {P}x{P}")

n_patches = (H // P) * (W // P)
patch_dim = P * P * C
print(f"Number of patches N = (H/P) x (W/P) = {H//P} x {W//P} = {n_patches}")
print(f"Flattened patch dimension = P*P*C = {P}*{P}*{C} = {patch_dim}\n")

# 1. Split into patches and flatten
patches = []
for i in range(0, H, P):
    for j in range(0, W, P):
        patches.append(image[i:i+P, j:j+P, :].reshape(-1))
patches = np.array(patches)
print(f"Step 1 - patch matrix shape: {patches.shape}  (N x patch_dim)")

# 2. Linear projection to d dimensions
E = np.random.randn(patch_dim, d) * 0.05
embeddings = patches @ E
print(f"Step 2 - after linear projection E {E.shape}: {embeddings.shape}  (N x d)")

# 3. Prepend the learnable [CLS] token
cls_token = np.random.randn(1, d) * 0.05
seq = np.vstack([cls_token, embeddings])
print(f"Step 3 - prepend [CLS] token: {seq.shape}  (N+1 x d)")

# 4. Add position encodings
pos = np.random.randn(*seq.shape) * 0.05
seq = seq + pos
print(f"Step 4 - add position embeddings: {seq.shape}")

print(f"\nFinal sequence fed to the Transformer encoder: {seq.shape[0]} tokens of dim {d}")
print(f"  token 0      = [CLS]  (its final output is used for classification)")
print(f"  tokens 1..{n_patches} = image patches\n")
print(f"First 6 dims of [CLS] token : {np.round(seq[0][:6], 4)}")
print(f"First 6 dims of patch 1     : {np.round(seq[1][:6], 4)}")

print("\n=== Sequence length vs patch size ===\n")
print(f"{'Patch size':>11} {'#Patches':>10} {'Patch dim':>11}  {'Note'}")
print("-" * 62)
for p in [4, 8, 16]:
    n = (H//p) * (W//p)
    note = "more tokens = finer detail, slower" if p == 4 else (
           "fewer tokens = faster, coarser" if p == 16 else "balanced")
    print(f"{p:>7}x{p:<3} {n:>10} {p*p*C:>11}  {note}")

print("\nViT for image captioning: ViT encoder represents the image, then a")
print("Transformer decoder generates the caption word by word.")

=== Vision Transformer (ViT): 'An image is worth 16x16 words' ===

Idea: split the image into fixed-size patches, flatten each patch,
linearly project it -> now the image IS a sequence, so a Transformer
encoder can process it exactly like a sentence.

Input image : 32x32x3
Patch size  : 8x8
Number of patches N = (H/P) x (W/P) = 4 x 4 = 16
Flattened patch dimension = P*P*C = 8*8*3 = 192

Step 1 - patch matrix shape: (16, 192)  (N x patch_dim)
Step 2 - after linear projection E (192, 16): (16, 16)  (N x d)
Step 3 - prepend [CLS] token: (17, 16)  (N+1 x d)
Step 4 - add position embeddings: (17, 16)

Final sequence fed to the Transformer encoder: 17 tokens of dim 16
  token 0      = [CLS]  (its final output is used for classification)
  tokens 1..16 = image patches

First 6 dims of [CLS] token : [-0.0059  0.0408 -0.0926  0.0464  0.0997 -0.0908]
First 6 dims of patch 1     : [-0.3084 -0.2176 -0.3752  0.4417 -0.5824 -0.4827]

=== Sequence length vs patch size ===

 Patch size   #Patches   Pa

## Example 2: CLIP Contrastive Image-Caption Matching

This builds CLIP's training objective: encode images and captions into the same space, compute the cosine similarity of every pair, then maximise the N correct pairs and minimise the N^2 - N incorrect ones. The similarity matrix shows a strong diagonal, retrieval is 100% accurate, and the contrastive loss is computed in both directions.

In [2]:
# Chapter 27 - Example 2: CLIP-style contrastive image-caption matching
import numpy as np
np.random.seed(7)

print("=== CLIP: Contrastive Language-Image Pre-training ===\n")
print("Training objective:")
print("  1. Encode all images and all captions into the SAME m-dim space")
print("  2. Compute cosine similarity of every (image, caption) pair")
print("  3. MAXIMISE similarity of the N correct pairs")
print("     MINIMISE similarity of the N^2 - N incorrect pairs\n")

N, m = 4, 8
captions = ["a dog on a beach", "a plate of pasta",
            "a red sports car", "two children playing"]

# Simulate encoders that have already learned aligned representations
base = np.random.randn(N, m)
img_emb = base + 0.25 * np.random.randn(N, m)   # image encoder output
txt_emb = base + 0.25 * np.random.randn(N, m)   # text encoder output

def l2norm(M):
    return M / np.linalg.norm(M, axis=1, keepdims=True)

I, Tt = l2norm(img_emb), l2norm(txt_emb)
S = I @ Tt.T                     # cosine similarity matrix (N x N)

print(f"Similarity matrix S (rows = images, cols = captions):\n")
head = "            " + "".join(f"cap{j+1:<7}" for j in range(N))
print(head)
for i in range(N):
    row = "".join(f"{S[i,j]:+.3f}  " for j in range(N))
    print(f"  image{i+1}   {row}")

print(f"\nDiagonal (correct pairs)  : {np.round(np.diag(S), 3)}")
off = S[~np.eye(N, dtype=bool)]
print(f"Off-diagonal (wrong pairs): mean {off.mean():+.3f}")
print(f"Number of wrong pairs = N^2 - N = {N*N - N}")

# Retrieval accuracy
pred = S.argmax(axis=1)
acc = (pred == np.arange(N)).mean()
print(f"\nImage -> caption retrieval (pick argmax per row):")
for i in range(N):
    ok = "correct" if pred[i] == i else "WRONG"
    print(f"  image{i+1} -> '{captions[pred[i]]}'   [{ok}]")
print(f"Retrieval accuracy: {acc*100:.0f}%")

# Contrastive (InfoNCE) loss
tau = 0.07
def softmax_rows(M):
    e = np.exp(M - M.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)
P = softmax_rows(S / tau)
loss_i2t = -np.mean(np.log(np.diag(P) + 1e-12))
Pt = softmax_rows((S.T) / tau)
loss_t2i = -np.mean(np.log(np.diag(Pt) + 1e-12))
print(f"\nContrastive loss (temperature tau={tau}):")
print(f"  image->text loss = {loss_i2t:.4f}")
print(f"  text->image loss = {loss_t2i:.4f}")
print(f"  total = (sum)/2  = {(loss_i2t + loss_t2i)/2:.4f}")

print("\nCLIP is used in DALL-E 2: the CLIP text encoder maps the caption into")
print("representation space, then a diffusion prior maps it to a CLIP image")
print("encoding, and reverse diffusion generates the final image.")

=== CLIP: Contrastive Language-Image Pre-training ===

Training objective:
  1. Encode all images and all captions into the SAME m-dim space
  2. Compute cosine similarity of every (image, caption) pair
  3. MAXIMISE similarity of the N correct pairs
     MINIMISE similarity of the N^2 - N incorrect pairs

Similarity matrix S (rows = images, cols = captions):

            cap1      cap2      cap3      cap4      
  image1   +0.946  +0.577  -0.661  -0.390  
  image2   +0.660  +0.829  -0.059  +0.030  
  image3   -0.541  -0.233  +0.979  +0.742  
  image4   -0.394  -0.387  +0.703  +0.951  

Diagonal (correct pairs)  : [0.946 0.829 0.979 0.951]
Off-diagonal (wrong pairs): mean +0.004
Number of wrong pairs = N^2 - N = 12

Image -> caption retrieval (pick argmax per row):
  image1 -> 'a dog on a beach'   [correct]
  image2 -> 'a plate of pasta'   [correct]
  image3 -> 'a red sports car'   [correct]
  image4 -> 'two children playing'   [correct]
Retrieval accuracy: 100%

Contrastive loss (tempe

## Example 3: Retrieval Augmented Generation Pipeline

This runs the three RAG stages end to end: RETRIEVE the most similar documents from a document store by cosine similarity, AUGMENT the prompt by inserting them as context, and GENERATE an answer grounded in that context. It shows why RAG reduces hallucination - the answer is traceable to a specific retrieved document, and knowledge can be updated by changing the store rather than retraining the model.

In [3]:
# Chapter 27 - Example 3: Retrieval Augmented Generation (RAG) pipeline
import numpy as np
np.random.seed(11)

print("=== Retrieval Augmented Generation (RAG) ===\n")
print("Problem: an LLM's knowledge is frozen at training time and it may")
print("hallucinate. RAG fixes this by RETRIEVING relevant documents from a")
print("document store and putting them into the prompt before generation.\n")

# ---- Document store ----
documents = [
 "The Transformer model was introduced in the paper Attention Is All You Need in 2017.",
 "BERT is an encoder-only model pre-trained with masked language modelling.",
 "GPT is a decoder-only autoregressive model pre-trained on unlabeled text.",
 "ResNet uses residual skip connections to train very deep CNNs.",
 "K-Means is a partitioning clustering algorithm that minimises distortion.",
 "LSTM cells use forget, input and output gates to avoid vanishing gradients.",
]

# ---- Tiny bag-of-words embedder (stands in for a real sentence encoder) ----
vocab = sorted({w.lower().strip('.,') for doc in documents for w in doc.split()})
idx = {w: i for i, w in enumerate(vocab)}

def embed(text):
    v = np.zeros(len(vocab))
    for w in text.lower().split():
        w = w.strip('.,?')
        if w in idx:
            v[idx[w]] += 1
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

doc_emb = np.array([embed(d) for d in documents])
print(f"Document store : {len(documents)} documents")
print(f"Vocabulary size: {len(vocab)} terms")
print(f"Embedding matrix shape: {doc_emb.shape}\n")

# ---- Query ----
query = "Which model is pre-trained with masked language modelling?"
q = embed(query)
print(f"User query: '{query}'\n")

# ---- Step 1: Retrieve ----
sims = doc_emb @ q
order = np.argsort(-sims)
k = 2
print("Step 1 - RETRIEVE: cosine similarity of query with each document")
for r, i in enumerate(order, 1):
    mark = "  <-- retrieved" if r <= k else ""
    print(f"  {sims[i]:.4f}  doc{i+1}: {documents[i][:58]}...{mark}")

retrieved = [documents[i] for i in order[:k]]

# ---- Step 2: Augment the prompt ----
print(f"\nStep 2 - AUGMENT: build the prompt with the top-{k} documents")
prompt = "Context:\n"
for i, d in enumerate(retrieved, 1):
    prompt += f"  [{i}] {d}\n"
prompt += f"Question: {query}\nAnswer using only the context above."
print("-" * 68)
print(prompt)
print("-" * 68)

# ---- Step 3: Generate ----
print("\nStep 3 - GENERATE: the LLM answers grounded in the retrieved context")
print("  Answer: BERT is pre-trained with masked language modelling.")
print("          (grounded in retrieved document [1], not invented)")

print("\n=== Why RAG matters ===")
print("  * Knowledge can be updated by changing the document store,")
print("    with no model retraining required.")
print("  * Answers are traceable to source documents (less hallucination).")
print("  * Retrieval quality sets the ceiling: bad retrieval -> bad answer.")

=== Retrieval Augmented Generation (RAG) ===

Problem: an LLM's knowledge is frozen at training time and it may
hallucinate. RAG fixes this by RETRIEVING relevant documents from a
document store and putting them into the prompt before generation.

Document store : 6 documents
Vocabulary size: 56 terms
Embedding matrix shape: (6, 56)

User query: 'Which model is pre-trained with masked language modelling?'

Step 1 - RETRIEVE: cosine similarity of query with each document
  0.8367  doc2: BERT is an encoder-only model pre-trained with masked lang...  <-- retrieved
  0.3586  doc3: GPT is a decoder-only autoregressive model pre-trained on ...  <-- retrieved
  0.1734  doc1: The Transformer model was introduced in the paper Attentio...
  0.1260  doc5: K-Means is a partitioning clustering algorithm that minimi...
  0.0000  doc4: ResNet uses residual skip connections to train very deep C...
  0.0000  doc6: LSTM cells use forget, input and output gates to avoid van...

Step 2 - AUGMENT: build th